# 🤖 NB-16: Agent Tool-Use Fine-tuning with ReAct Framework
**Goal:** Fine-tune a model to produce ReAct-style (Thought→Action→Observation) agent traces from Claude's thinking patterns.

**Modality:** Agent/Tool-use | **Model:** T5 + custom agent head

In [ ]:
!pip install transformers datasets langchain -q

In [ ]:
import json, re, random
random.seed(42)

def load_dataset(path="claude-opus-4_5-250x_jsonl.txt"):
    """Load Claude thinking dataset from JSONL."""
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def extract_fields(records):
    data = []
    for r in records:
        msgs = r["messages"]
        user = next((m["content"] for m in msgs if m["role"]=="user"), "")
        asst = next((m["content"] for m in msgs if m["role"]=="assistant"), "")
        think = re.findall(r"<think>(.*?)</think>", asst, re.DOTALL)
        response = re.sub(r"<think>.*?</think>", "", asst, flags=re.DOTALL).strip()
        data.append({
            "user": user,
            "assistant_full": asst,
            "thinking": " ".join(think).strip(),
            "response": response
        })
    return data

records = load_dataset()
data = extract_fields(records)
print(f"Loaded {len(data)} examples")
print(f"Sample user: {data[0]['user'][:80]}")
print(f"Sample think: {data[0]['thinking'][:120]}...")
print(f"Sample resp: {data[0]['response'][:120]}...")


In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from datasets import Dataset
import torch, re

# Convert Claude thinking into ReAct traces
TOOLS = ["search", "calculator", "code_exec", "summarize", "lookup"]

def to_react(d):
    think = d["thinking"][:400] if d["thinking"] else "No explicit reasoning."
    resp  = d["response"][:300]
    # Infer action from keywords
    if any(k in d["user"].lower() for k in ["calculate","compute","math","sum"]):
        action = "calculator"
    elif any(k in d["user"].lower() for k in ["code","implement","write"]):
        action = "code_exec"
    elif any(k in d["user"].lower() for k in ["search","find","look up","recent"]):
        action = "search"
    else:
        action = "summarize"
    return (
        f"Question: {d['user']}",
        f"Thought: {think[:200]}\nAction: {action}\nObservation: [tool result]\nFinal Answer: {resp[:200]}"
    )

pairs = [to_react(d) for d in data]
src, tgt = zip(*pairs)

tok = T5Tokenizer.from_pretrained("google/flan-t5-small")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small")

def preprocess(batch):
    inp = tok(list(batch["src"]), max_length=256, truncation=True, padding="max_length")
    lab = tok(list(batch["tgt"]), max_length=256, truncation=True, padding="max_length")
    inp["labels"] = [[(t if t != tok.pad_token_id else -100) for t in l] for l in lab["input_ids"]]
    return inp

ds = Dataset.from_dict({"src":list(src),"tgt":list(tgt)}).map(preprocess, batched=True, remove_columns=["src","tgt"]).train_test_split(0.1)
args = Seq2SeqTrainingArguments("./agent-react", num_train_epochs=5,
    per_device_train_batch_size=8, predict_with_generate=True,
    fp16=torch.cuda.is_available(), evaluation_strategy="epoch", report_to="none")
Seq2SeqTrainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["test"],
    tokenizer=tok, data_collator=DataCollatorForSeq2Seq(tok, model)).train()
print("✅ ReAct agent model fine-tuned!")
